# RAG over Emails & Chat Messages

Real-world knowledge bases aren't just articles — they include emails, Slack messages, meeting notes, and other metadata-rich documents. This notebook shows how Neocortex handles:

- Documents with structured metadata (sender, recipient, date, subject)
- Multiple document sources indexed into the same graph
- Cross-source queries that synthesize information from emails and chats

All data in this notebook is synthetic — a coherent workplace narrative about a product launch at a fictional company.

In [ ]:
import os
import time

from dotenv import load_dotenv

load_dotenv()

from neocortex import GraphRAG, QueryParam
from neocortex._llm import token_tracker

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment ready.")

## Email Data

Eight emails tracking the "Project Aurora" product launch at NovaTech. Each email has metadata for sender, recipients, subject, and date.

In [ ]:
EMAILS = [
    (
        "Team, I'm excited to announce that Project Aurora has been greenlit by the board. "
        "We have a $2M budget and a target launch date of March 15, 2025. Sarah will lead "
        "the engineering team, and David will handle product marketing. Let's schedule a "
        "kickoff meeting for next Monday. - Rachel",
        {"from": "Rachel Kim", "to": "engineering-all@novatech.com", "subject": "Project Aurora - Approved!", "date": "2024-09-02", "source": "email"}
    ),
    (
        "Rachel, I've reviewed the Aurora technical requirements. We'll need to integrate "
        "with the existing Pulse API and add real-time analytics. I'm estimating 4 months "
        "for the backend work. My team (Jake, Priya, and Tom) can start immediately. One "
        "concern: the Pulse API has rate limiting issues we discovered last quarter — I'll "
        "need DevOps to provision dedicated endpoints. - Sarah",
        {"from": "Sarah Chen", "to": "Rachel Kim", "subject": "Re: Aurora Technical Assessment", "date": "2024-09-05", "source": "email"}
    ),
    (
        "Hi David, the brand guidelines for Aurora are attached. Key points: we're positioning "
        "this as an enterprise analytics platform. Target audience is VP-level decision makers "
        "at Fortune 500 companies. The competitive analysis shows our main rivals are DataViz Pro "
        "and InsightEngine. Our differentiator is the real-time graph-based analytics. "
        "Budget for launch marketing is $500K. - Rachel",
        {"from": "Rachel Kim", "to": "David Park", "subject": "Aurora Brand & Marketing Brief", "date": "2024-09-10", "source": "email"}
    ),
    (
        "Sarah, DevOps has provisioned the dedicated Pulse API endpoints. You should have "
        "access now. Also, I've flagged a potential security concern — the current auth "
        "token rotation for Pulse is every 24 hours, but Aurora's real-time features need "
        "persistent connections. I'm working with the security team on a solution. - Miguel",
        {"from": "Miguel Santos", "to": "Sarah Chen", "subject": "Pulse API Endpoints Ready + Security Note", "date": "2024-09-18", "source": "email"}
    ),
    (
        "All, quick update: we've completed the first sprint. The real-time data pipeline is "
        "working. Jake built the ingestion layer, Priya handled the graph transformation "
        "engine, and Tom set up the WebSocket streaming. We're on track for the alpha "
        "release in November. One blocker: we need UX designs for the dashboard. "
        "Lisa, can your team prioritize this? - Sarah",
        {"from": "Sarah Chen", "to": "engineering-all@novatech.com", "subject": "Aurora Sprint 1 Complete", "date": "2024-10-04", "source": "email"}
    ),
    (
        "Sarah, my team can start on the Aurora dashboard designs next week. I've assigned "
        "Alex and Nina to it. Based on the requirements doc, I'm proposing three dashboard "
        "views: executive summary, real-time monitor, and deep-dive analytics. I'll share "
        "wireframes by October 15. - Lisa",
        {"from": "Lisa Wang", "to": "Sarah Chen", "subject": "Re: Aurora UX Designs", "date": "2024-10-07", "source": "email"}
    ),
    (
        "Rachel, the alpha testing results are in. 15 internal users tested Aurora for two "
        "weeks. Overall satisfaction: 4.2/5. Main feedback: the executive dashboard is great, "
        "but users want more customization options for the deep-dive view. Performance is "
        "solid — 95th percentile latency is under 200ms for real-time updates. Two bugs "
        "filed: one rendering issue on Safari and one data sync delay. - Sarah",
        {"from": "Sarah Chen", "to": "Rachel Kim", "subject": "Aurora Alpha Results", "date": "2024-11-20", "source": "email"}
    ),
    (
        "Team, great news — our beta partner Meridian Corp has signed up for the Aurora pilot. "
        "They'll deploy it across their analytics department (40 users) starting December 5. "
        "Their CTO, Jennifer Walsh, will be our main point of contact. David, please coordinate "
        "the onboarding materials. Sarah, ensure the staging environment is ready. - Rachel",
        {"from": "Rachel Kim", "to": "engineering-all@novatech.com", "subject": "Meridian Corp Beta Partnership", "date": "2024-11-28", "source": "email"}
    ),
]

print(f"Loaded {len(EMAILS)} emails")

## Chat Data

Six Slack-style chat conversations from the same team, capturing informal discussions and quick updates.

In [ ]:
CHATS = [
    (
        "Jake: Hey Priya, the Pulse API rate limiter is acting up again. I'm seeing 429s "
        "on the batch ingestion endpoint.\n"
        "Priya: Same here. Miguel said the dedicated endpoints should fix this. Did you "
        "switch to the new URLs?\n"
        "Jake: Oh wait, I'm still on the old ones. Let me update the config.\n"
        "Jake: Fixed! Throughput is way better now — 10x improvement on batch calls.",
        {"channel": "#aurora-engineering", "participants": "Jake, Priya", "date": "2024-09-20", "source": "chat"}
    ),
    (
        "David: Just got off a call with the PR agency. They want to do a media embargo "
        "until Feb 1 and then a coordinated launch with TechCrunch and Wired.\n"
        "Rachel: Perfect. Make sure they highlight the real-time graph analytics angle. "
        "That's our key differentiator against DataViz Pro.\n"
        "David: Already on it. They also suggested we do a live demo at the Enterprise "
        "Analytics Summit in February.\n"
        "Rachel: Yes, book it. I'll present.",
        {"channel": "#aurora-marketing", "participants": "David, Rachel", "date": "2024-10-15", "source": "chat"}
    ),
    (
        "Tom: The WebSocket connection is dropping for users behind corporate proxies. "
        "I'm going to add a long-polling fallback.\n"
        "Sarah: Good call. How long will that take?\n"
        "Tom: About 3 days. I'll also add automatic reconnection with exponential backoff.\n"
        "Sarah: Ship it. We can't have the beta users losing real-time updates.",
        {"channel": "#aurora-engineering", "participants": "Tom, Sarah", "date": "2024-10-22", "source": "chat"}
    ),
    (
        "Lisa: @Sarah the wireframes are ready for review. Three views as discussed: "
        "executive summary, real-time monitor, and deep-dive. Alex did an amazing job "
        "on the real-time view — it has live-updating graph visualizations.\n"
        "Sarah: These look fantastic! One request: can we add a dark mode option? Several "
        "users mentioned it during the requirements gathering.\n"
        "Lisa: Sure, Nina can handle that. Two days tops.",
        {"channel": "#aurora-design", "participants": "Lisa, Sarah", "date": "2024-10-16", "source": "chat"}
    ),
    (
        "Miguel: Heads up — the security team approved the persistent connection model for "
        "Aurora. We're using mTLS certificates instead of rotating tokens for the WebSocket "
        "connections.\n"
        "Sarah: Excellent. Tom, you can update the connection layer.\n"
        "Tom: Already on it. I'll swap out the token auth by end of day.\n"
        "Miguel: I've also set up monitoring dashboards for the Aurora infrastructure. "
        "You can see them at internal.novatech.com/aurora-infra.",
        {"channel": "#aurora-engineering", "participants": "Miguel, Sarah, Tom", "date": "2024-10-25", "source": "chat"}
    ),
    (
        "Rachel: Just got off the phone with Jennifer Walsh at Meridian. They're very "
        "impressed with the beta so far. Their analytics team loves the real-time view. "
        "One request: they want to export graphs as PDF reports.\n"
        "Sarah: That's doable. Priya, can you add PDF export to the deep-dive view?\n"
        "Priya: Sure, I'll use the same renderer we built for the executive dashboard. "
        "Should have it ready by end of week.\n"
        "Rachel: Great. Jennifer also hinted they might want to expand to 200 users if the pilot goes well.",
        {"channel": "#aurora-general", "participants": "Rachel, Sarah, Priya", "date": "2024-12-10", "source": "chat"}
    ),
]

print(f"Loaded {len(CHATS)} chat conversations")

## Index Both Sources

We'll index emails and chat messages separately so we can see entity/relation counts from each source. Both go into the same knowledge graph.

In [ ]:
rag = GraphRAG(
    working_dir="./db/examples_emails",
    domain=(
        "workplace communications about a software product launch, including "
        "team members, project milestones, technical decisions, and business partnerships"
    ),
    example_queries=(
        "Who is working on Project Aurora?\n"
        "What technical challenges were encountered?\n"
        "What is the marketing strategy?\n"
        "Who are the beta partners?"
    ),
    entity_types=["person", "project", "company", "technology", "event"],
)

print("GraphRAG initialized.")

In [ ]:
# Index emails
email_texts = [text for text, _ in EMAILS]
email_metadata = [meta for _, meta in EMAILS]

token_tracker.reset()
start = time.perf_counter()

e_ent, e_rel, e_chunks = rag.insert(email_texts, metadata=email_metadata)

email_time = time.perf_counter() - start
email_cost = token_tracker.total_cost

print(f"Emails indexed in {email_time:.1f}s (${email_cost:.4f})")
print(f"  Entities: {e_ent} | Relations: {e_rel} | Chunks: {e_chunks}")

In [ ]:
# Index chat messages
chat_texts = [text for text, _ in CHATS]
chat_metadata = [meta for _, meta in CHATS]

token_tracker.reset()
start = time.perf_counter()

c_ent, c_rel, c_chunks = rag.insert(chat_texts, metadata=chat_metadata)

chat_time = time.perf_counter() - start
chat_cost = token_tracker.total_cost

print(f"Chats indexed in {chat_time:.1f}s (${chat_cost:.4f})")
print(f"  Entities: {c_ent} | Relations: {c_rel} | Chunks: {c_chunks}")
print()
print(f"Total: {e_ent + c_ent} entities, {e_rel + c_rel} relations, {e_chunks + c_chunks} chunks")

## Cross-Source Queries

These questions require synthesizing information from both emails and chats. Notice how the retrieved chunks span both sources.

In [ ]:
QUESTIONS = [
    "What technical challenges did the Aurora team face and how were they resolved?",
    "Who are all the people involved in Project Aurora and what are their roles?",
    "What is the timeline of Project Aurora from approval to beta launch?",
    "What feedback has been received about Aurora and from whom?",
]

for question in QUESTIONS:
    token_tracker.reset()
    start = time.perf_counter()

    response = rag.query(question, params=QueryParam.balanced())

    elapsed = time.perf_counter() - start

    print(f"\nQ: {question}")
    print(f"  ({elapsed:.1f}s | ${token_tracker.total_cost:.4f})")
    print(f"\nA: {response.response}")

    # Show which sources contributed
    sources = set()
    for chunk, score in response.context.chunks:
        source = chunk.metadata.get("source", "unknown")
        sources.add(source)
    print(f"\n  Sources used: {', '.join(sorted(sources))}")
    print("=" * 80)

## Key Takeaways

- **Metadata is preserved** through indexing and available on retrieved chunks, enabling source attribution.
- **Cross-source synthesis** works naturally — the knowledge graph connects entities regardless of which document they came from.
- **Incremental indexing** lets you add new data sources to an existing graph without rebuilding.
- Chunk metadata (sender, channel, date) helps trace answers back to their original context.

**Next steps:**
- [04_query_modes.ipynb](04_query_modes.ipynb) — Explore different query presets for quality/cost tradeoffs
- [05_multi_source_knowledge_base.ipynb](05_multi_source_knowledge_base.ipynb) — Workspace isolation for separate knowledge bases